# Load the Contoso Pharma lab data into the shared lakehouse (alternative)

The standard setup is UI-only: upload the Parquet files from `data/parquet` and use **Load to tables** (see `setup/README.md`, step 4).
Use this notebook instead if you prefer code. It loads the CSV files with explicit data types.

1. Upload the 6 CSV files from `data` to **Files/contoso_pharma** in `LH_ContosoPharma` (**Files** > **...** > **Upload** > **Upload files**).
2. Attach `LH_ContosoPharma` to this notebook as the default lakehouse (**Explorer** > **Add data items** > **From OneLake catalog**).
3. Select **Run all**. The notebook creates 6 Delta tables and prints row counts.

Optional: set `SOURCE = "github"` to download the CSV files from your public copy of the repository instead of uploading them.


In [ ]:
SOURCE = "files"  # "files" (uploaded CSV files) or "github"
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/CassandraOfTroy/fabric-semantic-model-data-agent-labs/main/data"
FILES_FOLDER = "Files/contoso_pharma"
LOCAL_FOLDER = "/lakehouse/default/Files/contoso_pharma"

In [ ]:
SCHEMAS = {
    "dim_customer": "customer_key INT, customer_name STRING, customer_type STRING, city STRING, country STRING, "
                    "country_code STRING, sales_region STRING, key_account STRING",
    "dim_product": "product_key INT, product_name STRING, brand STRING, therapeutic_area STRING, formulation STRING, "
                   "launch_date DATE, list_price_eur DECIMAL(12,2)",
    "dim_date": "date DATE, year INT, quarter STRING, year_quarter STRING, month_number INT, month_name STRING, "
                "year_month STRING, month_start_date DATE, week_ending_date DATE",
    "fact_sales": "sales_line_id BIGINT, order_date DATE, customer_key INT, product_key INT, channel STRING, units INT, "
                  "unit_price_eur DECIMAL(12,2), gross_sales_eur DECIMAL(14,2), discount_eur DECIMAL(14,2), net_sales_eur DECIMAL(14,2)",
    "inventory_weekly": "week_ending_date DATE, country STRING, country_code STRING, product_name STRING, brand STRING, "
                        "therapeutic_area STRING, stock_units INT, avg_weekly_demand_units INT, days_of_supply DOUBLE, "
                        "backorder_units INT, stock_status STRING",
    "quality_complaints": "complaint_id STRING, received_date DATE, country STRING, country_code STRING, product_name STRING, "
                          "brand STRING, therapeutic_area STRING, batch_number STRING, complaint_category STRING, "
                          "description STRING, severity STRING, status STRING, closed_date DATE, days_to_close INT",
}

In [ ]:
import os
import urllib.request

if SOURCE == "github":
    os.makedirs(LOCAL_FOLDER, exist_ok=True)
    for table in SCHEMAS:
        urllib.request.urlretrieve(f"{GITHUB_RAW_BASE}/{table}.csv", f"{LOCAL_FOLDER}/{table}.csv")
        print("downloaded", table)

In [ ]:
for table, schema in SCHEMAS.items():
    df = spark.read.option("header", True).schema(schema).csv(f"{FILES_FOLDER}/{table}.csv")
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table)
    print(f"{table:<20} {spark.table(table).count():>8,} rows")

## Check the result

Expected net sales: 2024 = 267.2, 2025 = 293.8, 2026 (January to September) = 215.1 (EUR million).

In [ ]:
spark.sql("""
    SELECT YEAR(order_date) AS year, ROUND(SUM(net_sales_eur) / 1e6, 1) AS net_sales_eur_m
    FROM fact_sales GROUP BY YEAR(order_date) ORDER BY year
""").show()